In [91]:
import numpy as np
import sys
sys.path.append("../../../")
sys.path.append("../../../Visualization/")
sys.path.append("../../../../")

In [92]:
experiment_file = '../../../experiments/parallelized_experiments/output/cosine_curve_amplitude_full_period/2023_12_17_15_52//experiment_result.json'

stiffness_path = '../../../experiments/parallelized_experiments/output/cosine_curve_amplitude_full_period/2023_12_17_15_52/'

### Overview

In [93]:
import pandas as pd
import json
import matplotlib.pyplot as plt
import numpy as np
import visualize_stiffness

In [94]:
with open(experiment_file, 'r') as fp:
    data = json.load(fp)

In [95]:
df = pd.DataFrame(data['data'])
valid_tags = np.array(df['name'][df['Planar equilibrium'] == 1])

In [96]:
kappa_path = None

In [97]:
name = 'cosine_curve_amplitude_full_period'

In [98]:
bending_stiffness_data, stretching_stiffness_data, scale_factor_data, used_tags = visualize_stiffness.plot_all_data(kappa_path, stiffness_path, name, valid_tags, plot_data = False)

In [99]:
parameters = (np.array(data['pattern_parameters'][0]['values']))

In [100]:
max_bending_stiffness = np.max(bending_stiffness_data, axis = 1)
min_bending_stiffness = np.min(bending_stiffness_data, axis = 1)
max_stretching_stiffness = np.max(stretching_stiffness_data, axis = 1)
min_stretching_stiffness = np.min(stretching_stiffness_data, axis = 1)

In [101]:
x_scale_factors, y_scale_factors = visualize_stiffness.get_axis_scale_factors(stiffness_path, name, valid_tags)

In [102]:
min_scale_factors = np.min(np.concatenate((x_scale_factors.reshape(-1, 1), y_scale_factors.reshape(-1, 1)), axis = 1), axis = 1)
max_scale_factors = np.max(np.concatenate((x_scale_factors.reshape(-1, 1), y_scale_factors.reshape(-1, 1)), axis = 1), axis = 1)

In [103]:
angle_offsets = visualize_stiffness.get_max_flattening_factor_offset(stiffness_path, name, valid_tags)

### Get scale function convex hull

In [104]:
import matplotlib.cm as cm
import matplotlib as mpl

In [105]:
from scipy.spatial import ConvexHull, convex_hull_plot_2d
import numpy as np
rng = np.random.default_rng()
points = rng.random((30, 2))   # 30 random points in 2-D
# points = np.concatenate((min_scale_factor.reshape((-1, 1)), max_scale_factor.reshape((-1, 1))), axis = 1)

points = np.concatenate((max_scale_factors.reshape((-1, 1)), min_scale_factors.reshape((-1, 1))), axis = 1)
hull = ConvexHull(points)

### Validate the max and min scale factors are aligned with the x and y axis

In [106]:
import parametrization_helper

In [107]:
eqns = hull.equations

In [108]:
parametrization_helper.visualize_scale_factors(eqns, max_scale_factors, min_scale_factors)

### Generate data without augmenting

In [109]:
import visualize_stiffness, importlib
importlib.reload(visualize_stiffness)

In [110]:
stiffness_coefficients = np.array(visualize_stiffness.get_stiffness_coefficients(stiffness_path, name, (used_tags)))
# For patches with reflection symmetry:
stiffness_coefficients[:, 1] *= 0
stiffness_coefficients[:, 2] *= 0

In [111]:
grid_data = np.zeros((9, len(parameters)))

In [112]:
for i in range(len(parameters)):
    grid_data[0][i] = max_scale_factors[i]
    grid_data[1][i] = min_scale_factors[i]
    grid_data[2][i] = x_scale_factors[i]
    grid_data[3][i] = y_scale_factors[i]
    for s in range(5):
        grid_data[4 + s][i] = stiffness_coefficients[i][s]

In [113]:
importlib.reload(parametrization_helper)

In [114]:
splines = parametrization_helper.ndsplines_get_mat_params_over_pattern_params_grid_interpolation(grid_data, (parameters))

In [115]:
scale_factors_grid_data = np.zeros((2, len(parameters)))
for i in range(len(parameters)):
    scale_factors_grid_data[0][i] = x_scale_factors[i]
    scale_factors_grid_data[1][i] = y_scale_factors[i]
scale_factors_splines = parametrization_helper.ndsplines_get_mat_params_over_pattern_params_grid_interpolation(scale_factors_grid_data, (parameters))

### Parametrization

In [116]:
import sys; sys.path.append('../../../'); sys.path.append('../../../periodic_patches/'); sys.path.append('../../experiments/'); sys.path.append('../../../gmsh')
import inflation, sparse_matrices, mesh, numpy as np, importlib, pickle
import inflatables_parametrization as parametrization
from numpy.linalg import norm
from io_redirection import suppress_stdout
import visualization

In [117]:
sys.path.append('periodic_patches/')
sys.path.append('gmsh')

In [118]:
from periodic_simulation_setup import *

In [119]:
from py_newton_optimizer import NewtonOptimizerOptions

In [120]:
import MeshFEM, parallelism, benchmark, utils
parallelism.set_max_num_tbb_threads(32)
parallelism.set_gradient_assembly_num_threads(32)
parallelism.set_hessian_assembly_num_threads(32)

In [121]:
import utils, mesh_utilities
importlib.reload(utils)

In [202]:
target_surf = mesh.Mesh("../../../../SiggraphExamples/Meshes/taller_hill.obj")
target_surf.setVertices(utils.prototypeScaleNormalization(target_surf.vertices(), placeAtopFloor=False))
# target_surf = mesh_utilities.subdivide_loop(target_surf, 1)
name = "hill"

In [203]:
lines = np.array(eqns)

In [204]:
# Run some iterations of the local-global algorithm to ensure a good separation between singular values.
# This step can also be used as a prediction of the feasiblity of a design surface:
# if it is unable to nearly satisfy the singular value constraints,
# the surface is probably infeasible.
lg_21 = parametrization.LocalGlobalParametrizer(target_surf, parametrization.lscm(target_surf))

lg_21.alphaMin = 1.4
lg_21.alphaMax = np.pi / 2
print(lg_21.energy())
for i in range(1000): lg_21.runIteration()

print(lg_21.energy())
lg_21.runIteration()
print(lg_21.energy())

In [205]:
visualization.visualize(lg_21)

In [206]:
new_lines = np.array([[0, 1, -1.05], [1, 0, -np.pi / 2.], [0, -1, 0.95], [-1., 0, 1.4]])
             # , [0, -1, -1.1], [1, 0, -1.4], [-1, 0, np.pi / 2]]

In [207]:
parametrization_helper.visualize_scale_factors(new_lines, lg_21.getAlphas(), np.ones(len(lg_21.getAlphas())), )

In [208]:
lg = parametrization.LocalGlobalGenericParametrizer(target_surf, parametrization.lscm(target_surf))

lg.alphaMin = 1.4
lg.alphaMax = np.pi / 2

lg.betaMin = 1.0
lg.betaMax = 1.0

print(lg.energy())
for i in range(1000): lg.runIteration()

print(lg.energy())
lg.runIteration()
print(lg.energy())

In [209]:
importlib.reload(visualization)
visualization.visualize_both(lg)

### New local global with convex hull

In [210]:
lg = parametrization.LocalGlobalGenericParametrizer(target_surf, parametrization.lscm(target_surf))

lg.setLines(eqns)

lg.alphaMin = hull.min_bound[0]
lg.alphaMax = hull.max_bound[0]

lg.betaMin = hull.min_bound[1]
lg.betaMax = hull.max_bound[1]

print(lg.energy())
for i in range(1000): lg.runIteration()

print(lg.energy())
lg.runIteration()
print(lg.energy())

In [211]:
lg.alphaMin, lg.alphaMax, lg.betaMin, lg.betaMax

In [212]:
importlib.reload(visualization)
visualization.visualize_both(lg, show_main = True)

In [213]:
parametrization_helper.visualize_scale_factors(eqns, lg.getAlphas(), lg.getBetas())

### Pattern parameters optimization

In [214]:
default_pattern_params = [0.2]  * len(lg.getAlphas())

In [215]:
mat_info = np.array(default_pattern_params).reshape((1, len(lg.getAlphas())))

In [216]:
rparam = parametrization.RegularizedPatternParametrizer(lg, splines, default_pattern_params, len(grid_data.shape) - 1)
rparam.patternParamBounds = np.array([[0.03, 0.4]])
rparam.diffRegW = 0.0

In [217]:
visualization.visualize_both(rparam, height = 4)

In [218]:
PET = parametrization.RegularizedPatternParametrizer.PatternEnergyType

In [219]:
import parametrization_optimization_helper

In [220]:
rparam.bendRegW = 1

In [221]:
rparam.energy(PET.RGP)

In [222]:
rparam.energy(PET.Bending)

In [223]:
num_pattern_params = 1

In [224]:
parametrization_optimization_helper.initialize_pattern_parameters(rparam, lines, num_pattern_params)

In [225]:
phiRegW = parametrization_optimization_helper.add_phi_regularization(rparam, lines, num_pattern_params)

In [226]:
patternRegW = parametrization_optimization_helper.add_pattern_regularization(rparam, lines, num_pattern_params, phiRegW)

### Bending

In [227]:
bendRegW = parametrization_optimization_helper.add_bending_energy(rparam, lines, num_pattern_params, phiRegW, patternRegW)

## Upsampling and channel generation

In [228]:
import parametrization_helper
importlib.reload(parametrization_helper)

In [229]:
def fusing_curve_polyline(patternParams):
#     Draw cosine curves.
    amp = patternParams[0]
    def get_y_from_x(x):
        return amp * np.cos(x) * 0.5 * np.pi + np.pi / 2

    x_coords = np.linspace(-np.pi, np.pi, 15)
    y_coords = get_y_from_x(x_coords)
    x_coords += np.pi
    x_coords /= 2
    polyline = np.concatenate(((y_coords).reshape(-1, 1), (x_coords).reshape(-1, 1)), axis = 1)
    return [polyline]


In [230]:
sdfVertices, sdfTris, sdf, sheet_vxs, concatenated_polylines, sheet_edges_polylines,  boundaryVxs, boundaryEdges, upsampleMesh_vertices, upsampleMesh_triangles, upsampledAngles, upsampledPatternParams = parametrization_helper.get_polyline_from_pattern_parameters(rparam, fusing_curve_polyline, nsubdiv = 4, frequency=0.2, duplicates_removable_threshold=[1e-4, 1e-2, 1e-1, 1e0, 2e0])

In [231]:
if len(boundaryEdges) > 1:
    concatenated_boundary_edges = []
    for polyline in boundaryEdges:
        concatenated_boundary_edges.extend(polyline)
    concatenated_boundary_edges = np.array(concatenated_boundary_edges)


In [232]:
visualization.scalarFieldPlotFast(sdfVertices, sdfTris, sdf, width = 5, height=5)
visualization.plot_line_segments(sheet_vxs, concatenated_polylines, width = 5, height = 5)
# visualization.plot_line_segments(list(sheet_vxs) + list(boundaryVxs), list(concatenated_polylines) + list(concatenated_boundary_edges + len(sheet_vxs)), width = 5, height = 5)
# plt.scatter(boundaryVxs[concatenated_boundary_edges[:,0], 0], boundaryVxs[concatenated_boundary_edges[:,0], 1], c = np.arange(len(concatenated_boundary_edges[:, 0])), cmap = mpl.colormaps['Greys'])

## Meshing and inflation simulation

In [233]:
import mesher_helper
importlib.reload(mesher_helper)

In [234]:
import time
time_stamp = time.strftime("%Y_%m_%d_%H_%M")

In [235]:
np.save("sheet_vxs.npy", sheet_vxs)
np.save("concatenated_polylines.npy", concatenated_polylines)

In [236]:
selected_elements = [np.array(sublist)[:, 0] for sublist in boundaryEdges[1:]]
holes_vxs = boundaryVxs[selected_elements]

In [237]:
holes_vxs

In [238]:
importlib.reload(mesher_helper)

In [239]:
v, f, fusing_data = mesher_helper.generate_mesh_non_periodic(4, boundaryVxs[np.array(boundaryEdges[0])[:, 0]], holes_vxs, sheet_vxs, concatenated_polylines, gui = False)

In [240]:
import numpy as np
import copy

# Use the function
new_v, new_f, new_fusing_without_boundary = parametrization_helper.remove_dangling_vertices(v, f - 1, fusing_data)
m = MeshFEM.mesh.Mesh(new_v, new_f)
new_fusing = copy.copy(new_fusing_without_boundary)
new_fusing[m.boundaryVertices()] = True

In [241]:
fusing_data, new_fusing

In [242]:
# m, iwv, iwbv = sheet_meshing.newMeshingAlgorithm(sdfVertices, sdfTris, sdf, SV, SE, triArea=1e0)


In [243]:
visualization.plot_2d_mesh(m, pointList=np.where(np.array(new_fusing) == 1)[0], width=10, height=10)


In [244]:
def edge_cb(a, b):
    print(a, b)

In [245]:
import inflation
isheet = inflation.InflatableSheet(m, new_fusing)

### Save pattern

In [246]:
import shapely
channelMargin = 4

In [247]:
V = m.vertices()
polylines = isheet.fusedRegionBooleanIntersectSheetBoundary()
shapely_boundaryEdges = shapely.MultiLineString([V[p] for p in polylines])
#utils.save(boundaryEdges.buffer(channelMargin), 'test.pkl.gz')
bypasses = shapely_boundaryEdges.buffer(channelMargin)
if bypasses.geom_type == 'Polygon': bypasses = [bypasses] # we generally expect a multipolygon...
outerAirChannelPolygons = [shapely.ops.unary_union([shapely.Polygon(boundaryVxs[np.array(boundaryEdges[0])[:, 0]])] + list(bypasses.geoms))]

In [248]:
smart_polygon = outerAirChannelPolygons[0]

In [249]:
coords = np.array(smart_polygon.exterior.coords)

In [250]:
polylines = []
for polyline in sheet_edges_polylines:
    polyline = np.array(polyline)
    polylines.append(sheet_vxs[np.array(list(polyline[:, 0]) + list([polyline[-1, 1]]))][:, :2].tolist())
parametrization_helper.save_to_obj(coords[:-1], polylines, 'hill_{}_sheet_pattern_{}_margin_{}.obj'.format(name, time_stamp, channelMargin))

### End

In [251]:
from mesh_utilities import SurfaceSampler, tubeRemesh


paramSampler = SurfaceSampler(np.pad(rparam.uv(), [(0, 0), (0, 1)], 'constant'), target_surf.triangles())
liftedSheetPositions = paramSampler.sample(m.vertices(), target_surf.vertices())

isheet.setUninflatedDeformation(liftedSheetPositions.transpose())

isheet.getVars()

In [252]:
import py_newton_optimizer
niter = 2000
iterations_per_output = 10
opts = py_newton_optimizer.NewtonOptimizerOptions()
opts.useIdentityMetric = True
opts.beta = 1e-4
opts.gradTol = 1e-10
opts.niter = iterations_per_output

In [253]:
# Are the flat region causing a problem? They might not actually control the metric...
# Try replacing them with single wall...
# Analyze the actual stretching factor (much easier to do with skeleton walls)

from tri_mesh_viewer import TriMeshViewer
viewer = TriMeshViewer(isheet, width=768, height=640)
viewer.showWireframe(True)

viewer.show()

In [254]:
viewer.update(scalarField=utils.getStrains(isheet)[:, 0])    

In [255]:
isheet.setUseTensionFieldEnergy(True)

isheet.setUseHessianProjectedEnergy(False)

fixedVars, hessianShift = [], 1e-6

framerate = 20
def cb(it):
    if it % framerate == 0:
        viewer.update(scalarField=utils.getStrains(isheet)[:, 0])    

### First solve with low pressure to get out of indefinite state

In [256]:
isheet.pressure = 1e-5

In [257]:
opts.niter = 5

import time
cr = inflation.inflation_newton(isheet, fixedVars, opts, hessianShift = hessianShift, callback = cb)

### Then inflate

In [258]:
isheet.pressure = 5e-2

In [259]:
opts.niter = 2000
opts.gradTol = 1e-7

import time
benchmark.reset()
cr = inflation.inflation_newton(isheet, fixedVars, opts, hessianShift = hessianShift, callback = cb)
benchmark.report()

isheet.tensionStateHistogram()

In [260]:
# Plot maximum tensile strains in the sheet to verify the pressure is reasonable
from matplotlib import pyplot as plt
plt.hist(utils.getStrains(isheet)[:, 0], bins=1000);
plt.xlim(-0.04, 0.1);

In [261]:
import gzip

In [ ]:
pickle.dump(isheet,  gzip.open("igloo_pattern_optimized_{}_low_frequency_with_bending_high_resolution.pkl.gz".format(time_stamp), 'wb'))

### Design Optimization

In [183]:
# Reset the inflation and set up target-attraction forces
isheet.setUninflatedDeformation(liftedSheetPositions.transpose(), prepareRigidMotionPinConstraints=False)
targetAttractedSheet = inflation.TargetAttractedInflation(isheet, target_surf)
targetAttractedSheet.energy(targetAttractedSheet.EnergyType.Fitting)

In [184]:
targetAttractedSheet.targetSurfaceFitter().holdClosestPointsFixed = True
targetAttractedSheet.fittingWeight = 1e-5

In [185]:
# Are the flat region causing a problem? They might not actually control the metric...
# Try replacing them with single wall...
# Analyze the actual stretching factor (much easier to do with skeleton walls)

from tri_mesh_viewer import TriMeshViewer
ta_viewer = TriMeshViewer(targetAttractedSheet, width=768, height=640)
ta_viewer.showWireframe(True)

ta_viewer.show()

In [186]:
fixedVars, hessianShift = [], 1e-6

framerate = 20
def ta_cb(it):
    if it % framerate == 0:
        ta_viewer.update(scalarField=utils.getStrains(targetAttractedSheet.sheet())[:, 0])    

In [187]:
ta_cb(0)

In [188]:
# # Re-inflate, this time applying target-attraction forces.
# import time
# isheet.pressure = 5e-2
# benchmark.reset()
# for step in range(int(niter / iterations_per_output)):
#     cr = inflation.inflation_newton(targetAttractedSheet, fixedVars, opts)
#     if cr.numIters() < iterations_per_output: break
#     viewer.update()
#     time.sleep(0.05) # Allow some mesh synchronization time for pythreejs
# benchmark.report()


opts.niter = 2000
opts.gradTol = 1e-7

import time
benchmark.reset()
cr = inflation.inflation_newton(targetAttractedSheet, fixedVars, opts, hessianShift = hessianShift, callback = ta_cb)
benchmark.report()

isheet.tensionStateHistogram()

In [192]:
importlib.reload(visualization)
importlib.reload(sheet_optimizer)

In [193]:
# Set up the sheet optimizer
import sheet_optimizer, opt_config
origDesignMesh = isheet.mesh().copy()

sheet_opt = sheet_optimizer.PySheetOptimizer(targetAttractedSheet, fixedVars, renderMode=sheet_optimizer.RenderMode.PYTHREEJS,
                                             detActivationThreshold=0.9, detActivationThresholdTubeTri=0.5,
                                             originalDesignMesh=origDesignMesh, fusingCurveSmoothnessConfig=opt_config.FusingCurveSmoothnessParams(0.0, 0.0, 1.0, 1.0))

In [194]:
# Configure some more weights
sheet_opt.rso.compressionPenaltyWeight = 1e-6
fcs = sheet_opt.rso.fusingCurveSmoothness()
fcs.interiorWeight = 0.05

In [195]:
sheet_opt.flat_viewer.showWireframe()
sheet_opt.viewer()

In [196]:
# Run the optimization
sheet_opt.setSolver(sheet_optimizer.Solver.SCIPY)
sheet_opt.optimize()

In [197]:
# Lower the interior weight
fcs = sheet_opt.rso.fusingCurveSmoothness()
fcs.interiorWeight = 0.05

In [198]:
# Continue the optimization
sheet_opt.optimize()

In [199]:
utils.allGradientNorms(sheet_opt.rso)

In [200]:
utils.allEnergies(sheet_opt.rso)

In [201]:
# Remove the target-attraction force and recompute the equilibrium
targetAttractedSheet.fittingWeight = 1e-8
inflation.inflation_newton(targetAttractedSheet, sheet_opt.rso.fixedEquilibriumVars(), sheet_opt.opts)
viewer.update()

In [ ]:
# Save the full state for later reloading with `sheet_optimizer.load()`
sheet_opt.save('sheet_opt.pkl.gz')